**CI twin of `ch01-logreg-to-neuron.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt
import numpy as np

digits = load_digits()
print(f"{len(digits.images)} digits, each {digits.images[0].shape}")
print(digits.images[0].astype(int))

fig, ax = plt.subplots(figsize=(2.2, 2.2))
ax.imshow(digits.images[0], cmap="gray_r")
ax.set_title(f"label: {digits.target[0]}", fontsize=9)
ax.axis("off")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

mask = (digits.target == 0) | (digits.target == 1)
X, y = digits.data[mask], digits.target[mask]
print(f"{len(y)} zeros-and-ones")

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
print(f"held-out accuracy: {accuracy_score(yte, clf.predict(Xte)):.3f}")

In [ ]:
import math

w = clf.coef_[0]          # 64 connection strengths
b = clf.intercept_[0]     # the bias

x = Xte[0]                # one held-out digit (truth: a zero)
z = float(np.dot(w, x) + b)
output = 1 / (1 + math.exp(-z))

print(f"pre-activation z = {z:.3f}")
print(f"neuron output σ(z) = {output:.6f}")
print(f"sklearn's predict_proba = {clf.predict_proba(Xte[[0]])[0, 1]:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(2.6, 2.6))
im = ax.imshow(w.reshape(8, 8), cmap="coolwarm")
ax.set_title("red: votes '1'   blue: votes '0'", fontsize=8)
ax.axis("off")
fig.colorbar(im, shrink=0.8)
plt.show()

In [ ]:
w = clf.coef_[0]
b = clf.intercept_[0]
z = float(np.dot(w, Xte[0]) + b)
output = 1 / (1 + math.exp(-z))

run_tests([
    ("pre-activation of the first held-out digit", round(z, 3), -13.27),
    ("the neuron's verdict (a confident zero)", round(output, 3), 0.0),
    ("identity, not analogy", round(
        abs(output - float(clf.predict_proba(Xte[[0]])[0, 1])), 9), 0.0),
])

In [ ]:
import math

def neuron(x, w, b):
    z = sum(xi * wi for xi, wi in zip(x, w)) + b
    return 1 / (1 + math.exp(-z))

run_tests([
    ("perfect doubt at z = 0", neuron([1.0, 2.0], [0.5, -0.25], 0.0), 0.5),
    ("gentle yes at z = 1", round(neuron([2.0, 0.0], [1.0, 1.0], -1.0), 4),
     0.7311),
    ("strong no at z = -4", round(neuron([0.0], [3.0], -4.0), 4), 0.018),
    ("bias alone can fire it", neuron([0.0, 0.0], [9.9, 9.9], 0.0), 0.5),
], tol=1e-9)